In [1]:
from utils.experiment_utils import get_all_experiments_info, load_best_model
import torch
import os
import hydra
from omegaconf import DictConfig, OmegaConf

import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader

import ot

from datasets.lineage_tracing import LTSeqDataset

from geomloss import SamplesLoss

from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, r2_score

from utils.eval_utils import compute_mmd_distance

/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
lts = LTSeqDataset(seed=42)

loading cached adata from ./data/processed/adata_pca_50.h5ad  !!
loading cached clone sets from ./data/processed!
splitting 1218 clones into 609 train and 609 test


In [3]:
configs = get_all_experiments_info('/orcd/home/002/gokulg/orcd/scratch/CoupledDistributionEmbeddings/outputs/', False)

cfgs = [c for c in configs if 'lineage' in c['name']]

fm_models = [c for c in cfgs if 'Flow' in c['generator']]
energy_models = [c for c in cfgs if 'mmd' in c['config']['generator'].values()]
sw_models = [c for c in cfgs if 'swd' in c['config']['generator'].values()]

print('fm models: ', fm_models)
print('energy models: ', energy_models)
print('sw models: ', sw_models)

fm models:  [{'name': 'lineage_supervised_8f559753c149d773fa8c49a89dcdf1a9', 'dir': '/orcd/home/002/gokulg/orcd/scratch/CoupledDistributionEmbeddings/outputs/lineage_supervised_8f559753c149d773fa8c49a89dcdf1a9', 'config': {'dataset': {'_target_': 'datasets.lineage_tracing.LTSeqDataset', 'set_size': 100, 'min_cells': 3, 'data_shape': [50], 'root': '/orcd/home/002/gokulg/orcd/scratch/CoupledDistributionEmbeddings/data', 'seed': '${seed}'}, 'encoder': {'_target_': 'encoder.encoders.DistributionEncoderGNN', 'in_dim': '${dataset.data_shape[0]}', 'latent_dim': '${experiment.latent_dim}', 'hidden_dim': '${experiment.hidden_dim}', 'set_size': '${experiment.set_size}', 'layers': 2, 'fc_layers': 2}, 'model': {'_target_': 'layers.MLP', 'in_dims': [50, 1, 128, 128], 'hidden_dim': 512, 'out_dim': 50, 'layers': 4}, 'coupling': {'_target_': 'types.NoneType'}, 'generator': {'_target_': 'generator.flow_matching.FlowMatchingGenerator', 'model': '${model}', 'sigma': 0.5}, 'optimizer': {'_target_': 'torch

In [4]:
def load_model(cfg, path, device):
    enc = hydra.utils.instantiate(cfg['encoder'])
    gen = hydra.utils.instantiate(cfg['generator'])
    state = load_best_model(path)
    enc.load_state_dict(state['encoder_state_dict'])
    gen.load_state_dict(state['generator_state_dict'])
    enc.eval()
    gen.eval()
    enc.to(device)
    gen.to(device)
    return enc, gen

In [5]:
def sliced_wasserstein_distance(x, y, num_projections=50, p=2):
    d = x.shape[1]
    
    # Generate random directions on the unit sphere
    theta = torch.randn(d, num_projections, device=x.device)
    theta = theta / torch.norm(theta, dim=0, keepdim=True)
    
    # Project samples onto each direction
    x_proj = x @ theta  # (n, num_projections)
    y_proj = y @ theta  # (m, num_projections)
    
    # Sort projections
    x_sorted = torch.sort(x_proj, dim=0)[0]
    y_sorted = torch.sort(y_proj, dim=0)[0]
    
    # Compute 1D Wasserstein distances
    if x_sorted.shape[0] != y_sorted.shape[0]:
        min_size = min(x_sorted.shape[0], y_sorted.shape[0])
        x_quantiles = x_sorted[torch.linspace(0, x_sorted.shape[0]-1, min_size, device=x.device).long()]
        y_quantiles = y_sorted[torch.linspace(0, y_sorted.shape[0]-1, min_size, device=y.device).long()]
        wasserstein_dists = torch.mean(torch.abs(x_quantiles - y_quantiles) ** p, dim=0) ** (1/p)
    else:
        wasserstein_dists = torch.mean(torch.abs(x_sorted - y_sorted) ** p, dim=0) ** (1/p)
    
    # Average over all projections
    return torch.mean(wasserstein_dists)

loss_energy = SamplesLoss("energy")
# loss_mmd = SamplesLoss("gaussian", blur=1)

def evaluate_model(y_hat, y_true, model_name):
    """Compute energy distance, MMD, and Sliced Wasserstein."""
    energy = loss_energy(y_hat, y_true)
    mmd = compute_mmd_distance(y_hat, y_true)
    
    # Compute sliced wasserstein for each sample pair
    sw_distances = []
    for i in range(y_hat.shape[0]):
        sw = sliced_wasserstein_distance(y_hat[i], y_true[i], num_projections=50)
        sw_distances.append(sw.item())
    sw_distances = np.array(sw_distances)
    
    energy_mean = energy.mean().item()
    energy_sem = energy.std().item() / np.sqrt(energy.shape[0])
    
    mmd_mean = mmd.mean().item()
    mmd_sem = mmd.std().item() / np.sqrt(mmd.shape[0])
    
    sw_mean = sw_distances.mean()
    sw_sem = sw_distances.std() / np.sqrt(len(sw_distances))
    
    print(f"{model_name}")
    print(f"Energy Distance: {energy_mean:.6f} \pm {energy_sem:.6f}")
    print(f"MMD (RBF kernel):  {mmd_mean:.6f} \pm {mmd_sem:.6f}")
    print(f"Sliced Wasserstein: {sw_mean:.6f} \pm {sw_sem:.6f}")
    print()
    
    return {
        'energy': (energy_mean, energy_sem),
        'mmd': (mmd_mean, mmd_sem),
        'sliced_wasserstein': (sw_mean, sw_sem)
    }

In [7]:
from sklearn.linear_model import RidgeCV
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor


def fit_latent_predictor(method, X_train, Y_train, random_state=0):
    """fit a predictor mapping source latents -> target latents."""
    if method == 'ridge':
        alphas = np.logspace(-6, 6, 25)
        model = RidgeCV(alphas=alphas, cv=5, scoring='neg_mean_squared_error')
        model.fit(X_train, Y_train)
        info = {'alpha': float(model.alpha_)}

    elif method == 'mlp':
        model = MLPRegressor(
            hidden_layer_sizes=(256, 256),
            activation='relu',
            solver='adam',
            batch_size=256,
            learning_rate_init=1e-3,
            max_iter=500,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            random_state=random_state,
        )
        model.fit(X_train, Y_train)
        info = {'n_iter': int(model.n_iter_), 'final_loss': float(model.loss_)}

    elif method == 'rf':
        model = RandomForestRegressor(
            n_estimators=200,
            max_depth=None,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=random_state,
        )
        model.fit(X_train, Y_train)
        info = {'n_estimators': model.n_estimators}

    else:
        raise ValueError(f"unknown predictor method: {method}")

    return model, info


def evaluate_model_pair(model_configs, lts, device='cuda',
                        predictor_methods=('ridge', 'mlp', 'rf')):
    """
    evaluate a triplet of models (semisupervised, supervised, oracle).

    For the semi-supervised condition, fits each predictor in
    `predictor_methods` to map source latents to target latents and
    evaluates generation quality with each.
    """

    enc_semi, gen_semi = load_model(
        model_configs['semisupervised']['config'],
        model_configs['semisupervised']['dir'],
        device,
    )
    enc_sup, gen_sup = load_model(
        model_configs['supervised']['config'],
        model_configs['supervised']['dir'],
        device,
    )

    # encode everything once
    with torch.no_grad():
        z_x_train = enc_semi(lts.train_srcs.to(device))
        z_y_train = enc_semi(lts.train_tgts.to(device))
        z_x_test  = enc_semi(lts.test_srcs.to(device))
        z_y_test  = enc_semi(lts.test_tgts.to(device))
        z_x_sup_test = enc_sup(lts.test_srcs.to(device))

    X_train = z_x_train.cpu().numpy()
    Y_train = z_y_train.cpu().numpy()
    X_test  = z_x_test.cpu().numpy()
    Y_test  = z_y_test.cpu().numpy()

    src_test = lts.test_srcs
    tgt_test = lts.test_tgts

    # ---- conditions independent of the latent predictor ----
    # oracle: true target embeddings
    y_hat_oracle = gen_semi.sample(
        src_test.to(device).reshape(-1, 50),
        z_x_test.to(device),
        z_y_test.to(device),
    ).reshape(tgt_test.shape)

    # supervised baseline
    y_hat_sup = gen_sup.sample(
        src_test.to(device).reshape(-1, 50),
        z_x_sup_test.to(device),
        torch.zeros(z_x_test.shape).to(device),
    ).reshape(tgt_test.shape)

    model_name = model_configs['name']
    if model_name == 'DirectGenerator':
        model_name = model_configs['supervised']['config']['generator']['loss_type']

    results = {
        'model_name': model_name,
        'oracle': evaluate_model(
            y_hat_oracle.to(device), tgt_test.to(device),
            f"{model_name} - Oracle",
        ),
        'supervised': evaluate_model(
            y_hat_sup.to(device), tgt_test.to(device),
            f"{model_name} - Supervised",
        ),
        'predictors': {},
    }

    # ---- one semi-supervised eval per latent predictor ----
    for method in predictor_methods:
        print(f"  fitting {method} predictor...")
        predictor, info = fit_latent_predictor(method, X_train, Y_train)

        Y_pred_test = predictor.predict(X_test)
        mse = mean_squared_error(Y_test, Y_pred_test)
        r2  = r2_score(Y_test, Y_pred_test, multioutput='variance_weighted')

        y_hat_semi = gen_semi.sample(
            src_test.to(device).reshape(-1, 50),
            z_x_test.to(device),
            torch.tensor(Y_pred_test, dtype=torch.float32).to(device),
        ).reshape(tgt_test.shape)

        results['predictors'][method] = {
            'info': info,
            'latent_mse': float(mse),
            'latent_r2': float(r2),
            'semisupervised': evaluate_model(
                y_hat_semi.to(device), tgt_test.to(device),
                f"{model_name} - Semi-supervised ({method})",
            ),
        }

    return results


# organize configs into pairs
def pair_models(model_list):
    
    semi_models = [m for m in model_list if 'semisupervised' in m['name']]
    sup_models = [m for m in model_list if '_supervised' in m['name']]

    print(semi_models)
    
    if len(semi_models) > 1 or len(sup_models) > 1:
        print('warning multiple configs found, taking first only')
    pair = {'name': semi_models[0]['generator'].split('.')[-1],
            'semisupervised': semi_models[0],
            'supervised': sup_models[0]} 
    
    return [pair]


fm_pairs = pair_models(fm_models)
energy_pairs = pair_models(energy_models)
sw_pairs = pair_models(sw_models)

all_pairs = fm_pairs + energy_pairs + sw_pairs

print(f"found {len(all_pairs)} model pairs to evaluate:")
for pair in all_pairs:
    print(f"  - {pair['name']}: {pair['semisupervised']['name']} and {pair['supervised']['name']}")
print()

# Run evaluation on all pairs
all_results = []
for pair in all_pairs:
    print(f"evaluating {pair['name']}")
    results = evaluate_model_pair(pair, lts, device='cuda')
    all_results.append(results)
    print()

[{'name': 'lineage_semisupervised_43320c8fa9a07dd024cf61c9851479e7', 'dir': '/orcd/home/002/gokulg/orcd/scratch/CoupledDistributionEmbeddings/outputs/lineage_semisupervised_43320c8fa9a07dd024cf61c9851479e7', 'config': {'dataset': {'_target_': 'datasets.lineage_tracing.LTSeqDatasetUnstructured', 'set_size': 100, 'min_cells': 3, 'data_shape': [50], 'root': '/orcd/home/002/gokulg/orcd/scratch/CoupledDistributionEmbeddings/data', 'seed': '${seed}'}, 'encoder': {'_target_': 'encoder.encoders.DistributionEncoderGNN', 'in_dim': '${dataset.data_shape[0]}', 'latent_dim': '${experiment.latent_dim}', 'hidden_dim': '${experiment.hidden_dim}', 'set_size': '${experiment.set_size}', 'layers': 2, 'fc_layers': 2}, 'model': {'_target_': 'layers.MLP', 'in_dims': [50, 1, 128, 128], 'hidden_dim': 512, 'out_dim': 50, 'layers': 4}, 'coupling': {'_target_': 'types.NoneType'}, 'generator': {'_target_': 'generator.flow_matching.FlowMatchingGenerator', 'model': '${model}', 'sigma': 0.5}, 'optimizer': {'_target_'

In [8]:
def gen_label(name):
    if 'Flow' in name:
        return 'Flow matching'
    if 'mmd' in name:
        return 'Energy'
    return 'Sliced Wasserstein'


def fmt_mean_sem(t):
    return f"${t[0]:.3f} \\pm {t[1]:.3f}$"


# pretty names for predictors in the Type column
predictor_pretty = {'ridge': 'Ridge', 'mlp': 'MLP', 'rf': 'Random forest'}

rows = []
for r in all_results:
    g = gen_label(r['model_name'])

    # oracle and supervised: one row each
    for cond_key, cond_name in [('oracle', 'Oracle'), ('supervised', 'Supervised')]:
        m = r[cond_key]
        rows.append({
            'Generator': g,
            'Type': cond_name,
            'Energy distance': fmt_mean_sem(m['energy']),
            'SWD': fmt_mean_sem(m['sliced_wasserstein']),
            'MMD (RBF)': fmt_mean_sem(m['mmd']),
        })

    # one row per predictor for semi-supervised
    for method, pred_results in r['predictors'].items():
        m = pred_results['semisupervised']
        rows.append({
            'Generator': g,
            'Type': f"Semi-supervised ({predictor_pretty.get(method, method)})",
            'Energy distance': fmt_mean_sem(m['energy']),
            'SWD': fmt_mean_sem(m['sliced_wasserstein']),
            'MMD (RBF)': fmt_mean_sem(m['mmd']),
        })

df = pd.DataFrame(rows)

gen_order = ['Energy', 'Sliced Wasserstein', 'Flow matching']
cond_order = [
    'Oracle',
    'Supervised',
    'Semi-supervised (Ridge)',
    'Semi-supervised (MLP)',
    'Semi-supervised (Random forest)',
]

df['Generator'] = pd.Categorical(df['Generator'], categories=gen_order, ordered=True)
df['Type'] = pd.Categorical(df['Type'], categories=cond_order, ordered=True)
df = df.sort_values(['Generator', 'Type']).reset_index(drop=True)

# multiindex for generator groups
df_indexed = df.set_index(['Generator', 'Type'])

display(df_indexed.style.set_table_styles([
    {'selector': 'th', 'props': [('font-weight', 'bold'), ('text-align', 'center')]},
    {'selector': 'td', 'props': [('text-align', 'center')]},
]))

# latex with multirow
latex = df_indexed.to_latex(escape=False, multirow=True, column_format='llccc')
print(latex)

\begin{tabular}{llccc}
\toprule
 &  & Energy distance & SWD & MMD (RBF) \\
Generator & Type &  &  &  \\
\midrule
\multirow[t]{5}{*}{Energy} & Oracle & $1.440 \pm 0.022$ & $0.958 \pm 0.011$ & $2.881 \pm 0.045$ \\
 & Supervised & $4.709 \pm 0.127$ & $1.639 \pm 0.022$ & $9.419 \pm 0.255$ \\
 & Semi-supervised (Ridge) & $4.269 \pm 0.103$ & $1.579 \pm 0.020$ & $8.538 \pm 0.205$ \\
 & Semi-supervised (MLP) & $4.221 \pm 0.102$ & $1.569 \pm 0.020$ & $8.441 \pm 0.204$ \\
 & Semi-supervised (Random forest) & $4.757 \pm 0.127$ & $1.672 \pm 0.024$ & $9.514 \pm 0.254$ \\
\cline{1-5}
\multirow[t]{5}{*}{Sliced Wasserstein} & Oracle & $1.494 \pm 0.025$ & $0.953 \pm 0.011$ & $2.988 \pm 0.049$ \\
 & Supervised & $4.936 \pm 0.133$ & $1.676 \pm 0.023$ & $9.872 \pm 0.267$ \\
 & Semi-supervised (Ridge) & $4.333 \pm 0.108$ & $1.578 \pm 0.020$ & $8.666 \pm 0.215$ \\
 & Semi-supervised (MLP) & $4.374 \pm 0.109$ & $1.583 \pm 0.022$ & $8.748 \pm 0.218$ \\
 & Semi-supervised (Random forest) & $4.671 \pm 0.131$ & 